# Özdeğer ve Özvektör Hesaplamaları (Eigenvalues & Eigenvectors)

Bu notebook, bir matrisin özdeğer ve özvektörlerinin hem **sıfırdan (from scratch)** (QR algoritması ve Ters İterasyon kullanılarak) hem de **NumPy hazır fonksiyonu** (`numpy.linalg.eig`) kullanılarak hesaplanmasını, ardından bu iki yöntemin sonuçlarının karşılaştırılmasını içermektedir.

**Referans (Sıfırdan Algoritma Yaklaşımı):** [LucasBN/Eigenvalues-and-Eigenvectors](https://github.com/LucasBN/Eigenvalues-and-Eigenvectors)

## 1. Test Matrisinin Tanımlanması

Analizimiz boyunca üzerinde çalışacağımız $3 \times 3$ boyutlarındaki karesel test matrisi aşağıdaki gibi tanımlanmıştır:

In [ ]:
import numpy as np
import pandas as pd

# 3x3 Test Matrisi (Simetrik, reel özdeğerler garantili)
A = np.array([[4, 1, 1],
              [1, 3, 2],
              [1, 2, 5]], dtype=float)

print("Test Matrisi A:")
print(A)

## 2. Sıfırdan Hesaplama: QR Algoritması ve Ters İterasyon

Özdeğerleri bulmak için **QR Algoritması**'nı kullanıyoruz. Bu işlemde matrisin QR ayrışımını hesaplamak amacıyla **Gram-Schmidt Dikleştirme (Orthogonalization)** süreci manual olarak tanımlanmıştır.

Özdeğerler bulunduktan sonra, bunlara karşılık gelen özvektörleri hesaplamak için **Ters İterasyon (Inverse Iteration)** algoritması uygulanmaktadır.

In [ ]:
def gram_schmidt(A):
    """
    Gram-Schmidt ortogonizasyon metodu ile QR ayrışımı hesaplar.
    """
    n = A.shape[1]
    Q = np.zeros_like(A, dtype=float)
    R = np.zeros((n, n), dtype=float)
    
    for j in range(n):
        v = A[:, j]
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

def eig_qr(A, num_iters=1000, tol=1e-10):
    """
    QR algoritmasını tekrarlayarak matrisin özdeğerlerini bulur.
    A_{k+1} = R_k * Q_k işlemi iteratif olarak tekrarlanır.
    """
    Ak = np.copy(A)
    
    for _ in range(num_iters):
        Q, R = gram_schmidt(Ak)
        Ak = R @ Q
        
        # Çapraz dışı elemanlar toleransın altındaysa iterasyonu kes
        off_diag = np.sum(np.abs(Ak)) - np.sum(np.abs(np.diag(Ak)))
        if off_diag < tol:
            break
            
    # Köşegen üzerindeki değerler özdeğerlerdir
    eigenvalues = np.diag(Ak)
    return eigenvalues

def inverse_iteration(A, mu, num_iters=1000, tol=1e-6):
    """
    Bulunan özdeğere (mu) karşılık gelen özvektörü ters iterasyon ile hesaplar.
    (A - mu*I) w = v denklemi çözülerek v yakınsatılır.
    """
    n = A.shape[0]
    v = np.random.rand(n)
    v /= np.linalg.norm(v)
    I = np.eye(n)
    
    for _ in range(num_iters):
        try:
            w = np.linalg.solve(A - mu * I, v)
        except np.linalg.LinAlgError:
            # Boyut sıfırlanmasını engellemek için küçük bir epsilon ekle
            w = np.linalg.solve(A - (mu + 1e-10) * I, v)
        
        v_next = w / np.linalg.norm(w)
        
        # Yakınsama kontrolü
        if np.linalg.norm(v_next - v) < tol or np.linalg.norm(v_next + v) < tol:
            v = v_next
            break
        v = v_next
    return v

# Özdeğerleri (ve özvektörleri) sıfırdan oluşturduğumuz fonksiyonlarla hesaplayalım
eigenvalues_qr = eig_qr(A)

# Karşılaştırma kolaylığı için büyükten küçüğe sıralayalım
sort_indices = np.argsort(eigenvalues_qr)[::-1]
eigenvalues_qr = eigenvalues_qr[sort_indices]

eigenvectors_inv = np.zeros_like(A)
for j, mu in enumerate(eigenvalues_qr):
    eigenvectors_inv[:, j] = inverse_iteration(A, mu)

print("---- Sıfırdan Hesaplanan Sonuçlar ----")
print("QR Özdeğerleri:", eigenvalues_qr)
print("Ters İterasyon Özvektörleri:\n", eigenvectors_inv)

## 3. NumPy ile Hesaplama (`numpy.linalg.eig`)

Aynı işlemler, Python'un makine öğrenmesi kütüphanelerindeki standart olan güçlü optimize edilmiş LAPACK tabanlı `numpy.linalg.eig` kullanılarak hesaplanır.

In [ ]:
# NumPy ile hesaplama
np_eigenvalues, np_eigenvectors = np.linalg.eig(A)

# Karşılaştırma kolaylığı için büyükten küçüğe sıralayalım
np_sort_indices = np.argsort(np_eigenvalues)[::-1]
np_eigenvalues = np_eigenvalues[np_sort_indices]
np_eigenvectors = np_eigenvectors[:, np_sort_indices]

print("---- NumPy Sonuçları ----")
print("NumPy Özdeğerleri:", np_eigenvalues)
print("NumPy Özvektörleri:\n", np_eigenvectors)

## 4. Sonuçların Karşılaştırılması (Tablo)

Pandas DataFrame kullanılarak sıfırdan yazılan algoritmalar ile NumPy linalg kütüphanesinin sonuçlarının farkları analiz edilmiştir.

In [ ]:
# Özvektörlerin yönü (işareti) algoritmaya göre değişebileceğinden,
# tam karşılaştırma için vektör işaretlerini düzenliyoruz (içiçe faz ayarı).
for i in range(A.shape[1]):
    if eigenvectors_inv[0, i] < 0:
        eigenvectors_inv[:, i] *= -1
    if np_eigenvectors[0, i] < 0:
        np_eigenvectors[:, i] *= -1

# Verileri Tablo Haline Getirme
comparison_data = {
    'QR Özdeğerleri': eigenvalues_qr,
    'NumPy Özdeğerleri': np_eigenvalues,
    'Fark (Özdeğer)': np.abs(eigenvalues_qr - np_eigenvalues),
    'QR Özvektörü': [str(np.round(eigenvectors_inv[:, i], 4)) for i in range(3)],
    'NumPy Özvektörü': [str(np.round(np_eigenvectors[:, i], 4)) for i in range(3)]
}

df = pd.DataFrame(comparison_data)
df.index = [f"Eigen {i+1}" for i in range(3)]

print("Sıfırdan QR/Ters İterasyon ile NumPy Sonuçlarının Karşılaştırma Tablosu:")
df

## 5. Doğrulama: $A v = \lambda v$ Eşitliğinin Kontrolü

Hesaplanan bir özdeğer-özvektör çiftinin doğruluğu $A v = \lambda v$ denklemi ile kanıtlanır. Hesapladığımız sonuçların doğruluğunu bu kontrolle yapalım.

In [ ]:
print("===== DOĞRULAMA (A*v = lambda*v) =====")
for i in range(len(eigenvalues_qr)):
    v_custom = eigenvectors_inv[:, i]
    lam_custom = eigenvalues_qr[i]
    
    # Matris çarpımı (A * v)
    Av = A @ v_custom
    
    # Skaler çarpım (lambda * v)
    lam_v = lam_custom * v_custom
    
    print(f"\n► {i+1}. ÖZVEKTÖR ve ÖZDEĞER (Sıfırdan Hesaplanan)")
    print(f"A @ v     :  {np.round(Av, 6)}")
    print(f"lambda * v:  {np.round(lam_v, 6)}")
    
    is_close = np.allclose(Av, lam_v, atol=1e-5)
    print(f"Eşitlik sağlanıyor mu (1e-5 toleransla)? -> {is_close}")
